In [1]:
!apt-get install -y p7zip-full -qq
!pip install -q segmentation-models-pytorch albumentations tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 9.3 MB/s eta 0:00:00


### CELL 2 — Locate + extract dataset (adjust KAGGLE_DATASET_SLUG to match yours) ###


In [8]:
import os
 
# TODO: replace with your actual dataset slug, visible in the right-side "Input" panel
# once you've added your uploaded dataset to this notebook. Typically looks like:
# /kaggle/input/your-dataset-name/
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/zakariaassadiki/vessel-dataset"
LOCAL_ROOT = "/kaggle/working/vessel_data"
 
os.makedirs(LOCAL_ROOT, exist_ok=True)
 
# Check what's actually in the input dir first — if you uploaded as .zip, Kaggle
# already extracted it and you can skip the 7z extraction step entirely.
print("Contents of Kaggle input dir:")
for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    print(root, "->", files[:5], f"({len(files)} files)")
 
# If you uploaded the raw .7z (not pre-zipped), extract it here:
VESSEL_7Z_PATH = os.path.join(KAGGLE_INPUT_DIR, "vessel_dataset.7z")  # adjust filename
if os.path.exists(VESSEL_7Z_PATH):
    !7z x "{VESSEL_7Z_PATH}" -o{LOCAL_ROOT} -y
else:
    # already extracted by Kaggle (uploaded as .zip) — just point LOCAL_ROOT at the input dir directly
    LOCAL_ROOT = KAGGLE_INPUT_DIR
    print(f"No .7z found — assuming Kaggle already extracted your zip. Using LOCAL_ROOT = {LOCAL_ROOT}")

Contents of Kaggle input dir:
/kaggle/input/datasets/zakariaassadiki/vessel-dataset -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/masks -> ['mask_1713.tiff', 'mask_3905.tiff', 'mask_990.tiff', 'mask_2638.tiff', 'mask_5545.tiff'] (6544 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/patches -> ['patch_469.tiff', 'patch_2882.tiff', 'patch_4574.tiff', 'patch_2245.tiff', 'patch_3557.tiff'] (6544 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing -> [] (0 files)
/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_masks -> ['mask_291.tiff', 'mask_37.tiff', 'mask_216.tiff', 'mask_183.tiff', 'mask_539.tiff'] (728 files)
/kaggle/input/

### CELL 3 — Inspect extracted structure (RUN THIS AND READ THE OUTPUT BEFORE CONTINUING) ###


In [9]:
for root, dirs, files in os.walk(LOCAL_ROOT):
    depth = root.replace(LOCAL_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/  ({len(files)} files)")
    if files:
        print(f"{indent}  sample: {files[:5]}")
    if depth > 2:  # don't spam if it's deep
        continue
 
# Also print the raw file extension breakdown, useful to know what image/mask format we have
from collections import Counter
ext_counter = Counter()
for root, dirs, files in os.walk(LOCAL_ROOT):
    for f in files:
        ext_counter[os.path.splitext(f)[1].lower()] += 1
print("\nExtension counts:", ext_counter)

vessel-dataset/  (0 files)
  VascularModelDataset/  (0 files)
    Training/  (0 files)
      masks/  (6544 files)
        sample: ['mask_1713.tiff', 'mask_3905.tiff', 'mask_990.tiff', 'mask_2638.tiff', 'mask_5545.tiff']
      patches/  (6544 files)
        sample: ['patch_469.tiff', 'patch_2882.tiff', 'patch_4574.tiff', 'patch_2245.tiff', 'patch_3557.tiff']
    Testing/  (0 files)
      testset_masks/  (728 files)
        sample: ['mask_291.tiff', 'mask_37.tiff', 'mask_216.tiff', 'mask_183.tiff', 'mask_539.tiff']
      testset_images/  (728 files)
        sample: ['patch_469.tiff', 'patch_242.tiff', 'patch_620.tiff', 'patch_710.tiff', 'patch_228.tiff']
    WSI Annotations/  (4 files)
      sample: ['SSES2021 10 LIVER 12 7 21_J-21-153_4_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSES2021 14 LIVER LL 14 7 21_J-21-156_4_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSES2021 12 LIVER LL 13 7 21_J-21-155_6_Pig_GS 1-10000 + PSR_RUN12__Liver LL 0h_VAB.ndpi.ndpa', 'SSE

### CELL 4 — Build (image, mask) file pairs ###


In [10]:
# Real structure confirmed from Cell 3's output:
#   vessel_data/VascularModelDataset/Training/patches/patch_N.tiff
#   vessel_data/VascularModelDataset/Training/masks/mask_N.tiff
#   vessel_data/VascularModelDataset/Testing/testset_images/patch_N.tiff
#   vessel_data/VascularModelDataset/Testing/testset_masks/mask_N.tiff
# Pairing is exact: same numeric ID, different prefix ("patch_" vs "mask_").
 
DATASET_ROOT = os.path.join(LOCAL_ROOT, "VascularModelDataset")
TRAINING_PATCHES_DIR = os.path.join(DATASET_ROOT, "Training", "patches")
TRAINING_MASKS_DIR = os.path.join(DATASET_ROOT, "Training", "masks")
TEST_IMAGES_DIR = os.path.join(DATASET_ROOT, "Testing", "testset_images")
TEST_MASKS_DIR = os.path.join(DATASET_ROOT, "Testing", "testset_masks")
 
import glob
import re
 
def pair_by_id(images_dir, masks_dir, img_prefix="patch", mask_prefix="mask"):
    pairs = []
    missing = []
    for img_path in sorted(glob.glob(os.path.join(images_dir, f"{img_prefix}_*.tiff"))):
        fname = os.path.basename(img_path)
        m = re.match(rf"{img_prefix}_(\d+)\.tiff", fname)
        if not m:
            continue
        num = m.group(1)
        mask_path = os.path.join(masks_dir, f"{mask_prefix}_{num}.tiff")
        if os.path.exists(mask_path):
            pairs.append((img_path, mask_path))
        else:
            missing.append(fname)
    if missing:
        print(f"WARNING: {len(missing)} images had no matching mask, e.g. {missing[:5]}")
    return pairs
 
training_pairs = pair_by_id(TRAINING_PATCHES_DIR, TRAINING_MASKS_DIR)
test_pairs = pair_by_id(TEST_IMAGES_DIR, TEST_MASKS_DIR)
 
print(f"Training pairs found: {len(training_pairs)}  (expect ~6544)")
print(f"Test pairs found: {len(test_pairs)}  (expect ~728)")
print("Example training pair:", training_pairs[0])
print("Example test pair:", test_pairs[0])

Training pairs found: 6544  (expect ~6544)
Test pairs found: 728  (expect ~728)
Example training pair: ('/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/patches/patch_1.tiff', '/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Training/masks/mask_1.tiff')
Example test pair: ('/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_images/patch_1.tiff', '/kaggle/input/datasets/zakariaassadiki/vessel-dataset/VascularModelDataset/Testing/testset_masks/mask_1.tiff')


### CELL 5 — Train/val split (random, fixed seed — per project guide Section 3) ###


In [11]:
from sklearn.model_selection import train_test_split
 
train_pairs, val_pairs = train_test_split(training_pairs, test_size=0.2, random_state=42)
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}")

Train: 5235  Val: 1309


### CELL 5b — Inspect one mask file (confirms format before trusting Cell 6/7) ###


In [12]:
import cv2
import numpy as np
 
sample_mask = cv2.imread(training_pairs[0][1], cv2.IMREAD_UNCHANGED)
print("mask shape:", sample_mask.shape, "dtype:", sample_mask.dtype)
 
# mask is (H, W, 3) confirmed — get actual RGB color triplets, not flattened values.
# cv2 loads as BGR by default, so convert to RGB first for colors that match the
# paper's stated legend (blue=portal vein, green=central vein, red=artery, purple=bile duct).
sample_mask_rgb = cv2.cvtColor(sample_mask, cv2.COLOR_BGR2RGB)
unique_colors, counts = np.unique(sample_mask_rgb.reshape(-1, 3), axis=0, return_counts=True)
order = np.argsort(-counts)
print(f"{len(unique_colors)} unique RGB colors found. Top ones by pixel count:")
for idx in order[:10]:
    print(f"  RGB{tuple(unique_colors[idx])}  -> {counts[idx]} pixels")
 
# Repeat across a few more mask files, in case not all 5 classes appear in one patch
print("\nChecking a few more masks for full color inventory:")
all_colors = set()
for i in range(min(30, len(training_pairs))):
    m = cv2.imread(training_pairs[i][1], cv2.IMREAD_UNCHANGED)
    m_rgb = cv2.cvtColor(m, cv2.COLOR_BGR2RGB)
    colors = np.unique(m_rgb.reshape(-1, 3), axis=0)
    for c in colors:
        all_colors.add(tuple(int(v) for v in c))
print(f"Total distinct colors across 30 sample masks: {len(all_colors)}")
for c in sorted(all_colors):
    print(" ", c)

mask shape: (1024, 1024, 3) dtype: uint8
247 unique RGB colors found. Top ones by pixel count:
  RGB(np.uint8(0), np.uint8(0), np.uint8(0))  -> 853783 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(255))  -> 127124 pixels
  RGB(np.uint8(128), np.uint8(0), np.uint8(128))  -> 29293 pixels
  RGB(np.uint8(255), np.uint8(0), np.uint8(0))  -> 24514 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(253))  -> 492 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(252))  -> 469 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(1))  -> 417 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(2))  -> 379 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(254))  -> 349 pixels
  RGB(np.uint8(0), np.uint8(0), np.uint8(3))  -> 319 pixels

Checking a few more masks for full color inventory:
Total distinct colors across 30 sample masks: 813
  (0, 0, 0)
  (0, 0, 1)
  (0, 0, 2)
  (0, 0, 3)
  (0, 0, 4)
  (0, 0, 5)
  (0, 0, 6)
  (0, 0, 7)
  (0, 0, 8)
  (0, 0, 9)
  (0, 0, 10)
  (0, 0, 11)
  (0, 0, 12)
  (0, 0, 13)
  (0, 

### CELL 6 — Dataset class (vessel, 5-class, online augmentation toggle) ###


In [13]:
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
# Class value mapping, per the original paper's convention:
# 0 background, 1 portal vein, 2 central vein, 3 hepatic artery, 4 bile duct
NUM_CLASSES = 5
PATCH_SIZE = 256
 
def get_train_transforms(aug_on: bool):
    if aug_on:
        return A.Compose([
            A.Rotate(limit=180, border_mode=cv2.BORDER_REFLECT, p=0.7),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ElasticTransform(alpha=1, sigma=20, p=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.4),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        # aug OFF: normalize only, nothing stochastic (already resized in __getitem__)
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
 
def get_eval_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
 
class VesselDataset(Dataset):
    # Confirmed from Cell 5b: masks are RGB with 5 core colors + anti-aliased blends
    # at class boundaries. Nearest-color classification handles the blended edge
    # pixels by snapping each to its closest pure class color.
    REFERENCE_COLORS = np.array([
        [0, 0, 0],       # 0 background (black)
        [0, 0, 255],     # 1 portal vein (blue)
        [0, 255, 0],     # 2 central vein (green)
        [255, 0, 0],     # 3 artery (red)
        [128, 0, 128],   # 4 bile duct (purple)
    ], dtype=np.float32)
 
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform
 
    def __len__(self):
        return len(self.pairs)
 
    @staticmethod
    def rgb_mask_to_class_index(mask_rgb):
        h, w, _ = mask_rgb.shape
        flat = mask_rgb.reshape(-1, 3).astype(np.float32)
        # distance from every pixel to each of the 5 reference colors -> (N, 5)
        dists = np.linalg.norm(
            flat[:, None, :] - VesselDataset.REFERENCE_COLORS[None, :, :], axis=2
        )
        class_idx = np.argmin(dists, axis=1)
        return class_idx.reshape(h, w).astype(np.uint8)
 
    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
 
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # resize image to PATCH_SIZE here too, so it matches the mask's size below
        # before either goes into the albumentations pipeline (which checks that
        # image/mask shapes match before applying any transform).
        image = cv2.resize(image, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)
 
        mask_bgr = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask_rgb = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2RGB)
 
        # Resize the RGB mask down to patch size FIRST (nearest-neighbor preserves
        # discrete colors), THEN do the expensive per-pixel color matching — this
        # cuts the color-matching cost by (1024/256)^2 = 16x versus matching at
        # full resolution before resizing.
        mask_rgb_small = cv2.resize(mask_rgb, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = self.rgb_mask_to_class_index(mask_rgb_small)  # (PATCH_SIZE, PATCH_SIZE) uint8
 
        # image and mask are now both already at PATCH_SIZE — transform below only
        # does augmentation/normalization, no resizing needed.
        transformed = self.transform(image=image, mask=mask)
        image_t = transformed["image"]
        mask_t = transformed["mask"].long()
 
        return image_t, mask_t

In [19]:
CLASS_NAMES = ["background", "portal_vein", "central_vein", "artery", "bile_duct"]

class_pixel_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for img_path, mask_path in training_pairs[:50]:
    mask_bgr = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    mask_rgb = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2RGB)
    class_idx = VesselDataset.rgb_mask_to_class_index(mask_rgb)
    counts = np.bincount(class_idx.flatten(), minlength=NUM_CLASSES)
    class_pixel_counts += counts

total = class_pixel_counts.sum()
for c, name in enumerate(CLASS_NAMES):
    print(f"{name}: {100*class_pixel_counts[c]/total:.2f}%")

background: 89.63%
portal_vein: 5.91%
central_vein: 0.73%
artery: 2.74%
bile_duct: 0.99%


### CELL 7 — Loss: Dice + Focal (vessel, 5-class) ###


In [20]:
import torch.nn as nn
import torch.nn.functional as F
 
class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
 
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()
 
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
 
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()
 
class DiceFocalLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=0.5, focal_weight=0.5):
        super().__init__()
        self.dice = DiceLoss(num_classes)
        self.focal = FocalLoss()
        self.dw = dice_weight
        self.fw = focal_weight
 
    def forward(self, logits, targets):
        return self.dw * self.dice(logits, targets) + self.fw * self.focal(logits, targets)

### CELL 8 — Per-class Dice metric ###


In [21]:
def per_class_dice(logits, targets, num_classes, smooth=1e-6):
    preds = torch.argmax(logits, dim=1)
    dices = []
    for c in range(num_classes):
        pred_c = (preds == c).float()
        target_c = (targets == c).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dice_c = (2. * intersection + smooth) / (union + smooth)
        dices.append(dice_c.item())
    return dices  # list of length num_classes

### CELL 9 — Attention U-Net (custom implementation) ###


In [22]:
# segmentation_models_pytorch does not ship a literal "Attention U-Net", so this is
# a from-scratch implementation of Oktay et al. 2018's attention-gated U-Net:
# each skip connection passes through an attention gate that learns to suppress
# irrelevant background regions and emphasize the target structures before
# concatenating with the decoder features. No pretrained encoder here (built from
# scratch), so expect it to converge a bit slower per-epoch than the pretrained
# U-Net++ — factor that into how you read the comparison.
 
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x):
        return self.block(x)
 
class AttentionGate(nn.Module):
    """
    g: gating signal from the coarser decoder level
    x: skip connection features from the encoder
    Output: x reweighted by a learned attention map in [0, 1]
    """
    def __init__(self, gate_ch, skip_ch, inter_ch):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(gate_ch, inter_ch, 1),
            nn.BatchNorm2d(inter_ch),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(skip_ch, inter_ch, 1),
            nn.BatchNorm2d(inter_ch),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, 1),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)
 
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode="bilinear", align_corners=False)
        attn = self.relu(g1 + x1)
        attn = self.psi(attn)
        return x * attn
 
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=NUM_CLASSES, base_ch=32):
        super().__init__()
        chs = [base_ch, base_ch * 2, base_ch * 4, base_ch * 8, base_ch * 16]
 
        self.enc1 = ConvBlock(in_channels, chs[0])
        self.enc2 = ConvBlock(chs[0], chs[1])
        self.enc3 = ConvBlock(chs[1], chs[2])
        self.enc4 = ConvBlock(chs[2], chs[3])
        self.pool = nn.MaxPool2d(2)
 
        self.bottleneck = ConvBlock(chs[3], chs[4])
 
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2)
        self.att4 = AttentionGate(chs[3], chs[3], chs[3] // 2)
        self.dec4 = ConvBlock(chs[4], chs[3])
 
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2)
        self.att3 = AttentionGate(chs[2], chs[2], chs[2] // 2)
        self.dec3 = ConvBlock(chs[3], chs[2])
 
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2)
        self.att2 = AttentionGate(chs[1], chs[1], chs[1] // 2)
        self.dec2 = ConvBlock(chs[2], chs[1])
 
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2)
        self.att1 = AttentionGate(chs[0], chs[0], chs[0] // 2)
        self.dec1 = ConvBlock(chs[1], chs[0])
 
        self.out_conv = nn.Conv2d(chs[0], num_classes, 1)
 
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
 
        d4 = self.up4(b)
        e4_att = self.att4(d4, e4)
        d4 = self.dec4(torch.cat([d4, e4_att], dim=1))
 
        d3 = self.up3(d4)
        e3_att = self.att3(d3, e3)
        d3 = self.dec3(torch.cat([d3, e3_att], dim=1))
 
        d2 = self.up2(d3)
        e2_att = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))
 
        d1 = self.up1(d2)
        e1_att = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))
 
        return self.out_conv(d1)

### CELL 10 — Model factory ###


In [23]:
import segmentation_models_pytorch as smp
 
CLASS_NAMES = ["background", "portal_vein", "central_vein", "artery", "bile_duct"]
 
def build_model(architecture: str, num_classes: int = NUM_CLASSES):
    if architecture == "unetplusplus":
        return smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights="imagenet",
                                 in_channels=3, classes=num_classes)
    elif architecture == "attention_unet":
        return AttentionUNet(in_channels=3, num_classes=num_classes)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

### CELL 11 — Training loop ###


In [25]:
import time
import csv
import pandas as pd
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm
 
def train_run(run_name, architecture, aug_on, epochs, batch_size, lr=5e-4, max_batches=None):
    """
    run_name: e.g. "vessel_run1_augOFF"
    architecture: "unetplusplus" or "attention_unet"
    aug_on: bool
    max_batches: if set (e.g. 3), caps train/val batches per epoch — use this for a
                 quick smoke test to confirm the pipeline runs end-to-end before
                 committing to a full run. Leave as None for real training runs.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"=== {run_name} | arch={architecture} aug_on={aug_on} epochs={epochs} bs={batch_size} | device={device} ===")
 
    train_ds = VesselDataset(train_pairs, get_train_transforms(aug_on))
    val_ds = VesselDataset(val_pairs, get_eval_transforms())
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True, drop_last=True,
                               persistent_workers=True, prefetch_factor=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=True,
                             persistent_workers=True, prefetch_factor=2)
 
    model = build_model(architecture).to(device)
    criterion = DiceFocalLoss(NUM_CLASSES)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=2e-4)
    # cosine schedule T matched to actual epoch count — see project guide Section 7
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()
 
    checkpoint_dir = "/kaggle/working/checkpoints"
    log_dir = "/kaggle/working/logs"
    os.makedirs(checkpoint_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    log_path = f"{log_dir}/{run_name}.csv"
 
    log_rows = []
    start_time = time.time()
 
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} train", leave=False)
        for batch_idx, (images, masks) in enumerate(train_bar):
            if max_batches is not None and batch_idx >= max_batches:
                break
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")
        n_train_batches = min(len(train_loader), max_batches) if max_batches else len(train_loader)
        train_loss /= n_train_batches
 
        model.eval()
        val_loss = 0.0
        class_dice_sums = [0.0] * NUM_CLASSES
        n_batches = 0
        val_bar = tqdm(val_loader, desc=f"[{run_name}] epoch {epoch+1}/{epochs} val", leave=False)
        with torch.no_grad():
            for batch_idx, (images, masks) in enumerate(val_bar):
                if max_batches is not None and batch_idx >= max_batches:
                    break
                images, masks = images.to(device), masks.to(device)
                with autocast():
                    logits = model(images)
                    loss = criterion(logits, masks)
                val_loss += loss.item()
                dices = per_class_dice(logits, masks, NUM_CLASSES)
                for c in range(NUM_CLASSES):
                    class_dice_sums[c] += dices[c]
                n_batches += 1
        val_loss /= n_batches
        class_dice_avg = [s / n_batches for s in class_dice_sums]
 
        scheduler.step()
 
        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": scheduler.get_last_lr()[0],
        }
        for c, name in enumerate(CLASS_NAMES):
            row[f"dice_{name}"] = class_dice_avg[c]
        log_rows.append(row)
 
        print(f"epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | "
              f"dice(bg/pv/cv/art/bd) = {[round(d,3) for d in class_dice_avg]}")
 
        # checkpoint every 5 epochs + always on last epoch
        if (epoch + 1) % 5 == 0 or (epoch + 1) == epochs:
            ckpt_path = f"{checkpoint_dir}/{run_name}.pth"
            torch.save(model.state_dict(), ckpt_path)
 
        # write log every epoch (cheap, protects against disconnect losing history)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)
 
    total_time = time.time() - start_time
    print(f"=== {run_name} done in {total_time/60:.1f} min ===")
 
    return {
        "run_name": run_name,
        "architecture": architecture,
        "aug_on": aug_on,
        "epochs": epochs,
        "batch_size": batch_size,
        "final_val_loss": val_loss,
        "final_dice_overall": sum(class_dice_avg[1:]) / (NUM_CLASSES - 1),  # excl. background
        **{f"final_dice_{name}": class_dice_avg[c] for c, name in enumerate(CLASS_NAMES)},
        "train_time_min": total_time / 60,
    }

### CELL 12 — Run 1: U-Net++ baseline, augmentation OFF ###


In [26]:
result_run1 = train_run(
    run_name="vessel_run1_unetplusplus_augOFF",
    architecture="unetplusplus",
    aug_on=False,
    epochs=18,
    batch_size=16,
)

=== vessel_run1_unetplusplus_augOFF | arch=unetplusplus aug_on=False epochs=18 bs=16 | device=cuda ===


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

/tmp/ipykernel_58/2828639666.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[vessel_run1_unetplusplus_augOFF] epoch 1/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

...

[vessel_run1_unetplusplus_augOFF] epoch 17/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 17/18 | train_loss 0.1532 | val_loss 0.1828 | dice(bg/pv/cv/art/bd) = [0.976, 0.725, 0.775, 0.704, 0.602]


[vessel_run1_unetplusplus_augOFF] epoch 18/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

[vessel_run1_unetplusplus_augOFF] epoch 18/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 18/18 | train_loss 0.1496 | val_loss 0.1827 | dice(bg/pv/cv/art/bd) = [0.977, 0.73, 0.776, 0.7, 0.615]
=== vessel_run1_unetplusplus_augOFF done in 52.5 min ===


### CELL 13 — Run 2: U-Net++ baseline, augmentation ON ###


In [27]:
result_run2 = train_run(
    run_name="vessel_run2_unetplusplus_augON",
    architecture="unetplusplus",
    aug_on=True,
    epochs=18,
    batch_size=16,
)


=== vessel_run2_unetplusplus_augON | arch=unetplusplus aug_on=True epochs=18 bs=16 | device=cuda ===


/tmp/ipykernel_58/2828639666.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[vessel_run2_unetplusplus_augON] epoch 1/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

/tmp/ipykernel_58/2828639666.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[vessel_run2_unetplusplus_augON] epoch 1/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

...

[vessel_run2_unetplusplus_augON] epoch 17/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 17/18 | train_loss 0.1793 | val_loss 0.1707 | dice(bg/pv/cv/art/bd) = [0.98, 0.771, 0.777, 0.708, 0.622]


[vessel_run2_unetplusplus_augON] epoch 18/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

[vessel_run2_unetplusplus_augON] epoch 18/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 18/18 | train_loss 0.1820 | val_loss 0.1705 | dice(bg/pv/cv/art/bd) = [0.98, 0.769, 0.776, 0.707, 0.627]
=== vessel_run2_unetplusplus_augON done in 55.2 min ===


### CELL 14 — Run 3: Attention U-Net (architecture comparison), augmentation ON ###


In [28]:
# NOTE: AttentionUNet is trained from scratch (no pretrained encoder), unlike
# UnetPlusPlus which uses an ImageNet-pretrained ResNet34 encoder. It will likely
# need more epochs to reach comparable performance — with only 18 epochs, expect
# this run to look weaker than runs 1/2 partly for that reason, not purely
# architecture quality. Worth noting explicitly in your report (see Section 14 of
# the project guide re: honest framing of results).

result_run3 = train_run(
    run_name="vessel_run3_attentionunet_augON",
    architecture="attention_unet",
    aug_on=True,
    epochs=18,
    batch_size=16,  # drop to 8 if this OOMs
)

=== vessel_run3_attentionunet_augON | arch=attention_unet aug_on=True epochs=18 bs=16 | device=cuda ===


/tmp/ipykernel_58/2828639666.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


[vessel_run3_attentionunet_augON] epoch 1/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

/tmp/ipykernel_58/2828639666.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[vessel_run3_attentionunet_augON] epoch 1/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

...

[vessel_run3_attentionunet_augON] epoch 17/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 17/18 | train_loss 0.2284 | val_loss 0.2104 | dice(bg/pv/cv/art/bd) = [0.973, 0.643, 0.733, 0.672, 0.592]


[vessel_run3_attentionunet_augON] epoch 18/18 train:   0%|          | 0/327 [00:00<?, ?it/s]

[vessel_run3_attentionunet_augON] epoch 18/18 val:   0%|          | 0/82 [00:00<?, ?it/s]

epoch 18/18 | train_loss 0.2282 | val_loss 0.2088 | dice(bg/pv/cv/art/bd) = [0.974, 0.65, 0.736, 0.67, 0.595]
=== vessel_run3_attentionunet_augON done in 53.9 min ===


### CELL 15 — Save your run log, then get results back to the shared Drive folder ###


In [29]:
runs_log_vessel = pd.DataFrame([result_run1, result_run2, result_run3])
runs_log_path = "/kaggle/working/runs_log_vessel.csv"
runs_log_vessel.to_csv(runs_log_path, index=False)
print(f"Saved to {runs_log_path}")
runs_log_vessel

Saved to /kaggle/working/runs_log_vessel.csv


,run_name,architecture,aug_on,epochs,batch_size,final_val_loss,final_dice_overall,final_dice_background,final_dice_portal_vein,final_dice_central_vein,final_dice_artery,final_dice_bile_duct,train_time_min
0,vessel_run1_unetplusplus_augOFF,unetplusplus,False,18,16,0.182670,0.705353,0.977347,0.730161,0.776137,0.700484,0.614629,52.518072
1,vessel_run2_unetplusplus_augON,unetplusplus,True,18,16,0.170468,0.719640,0.980165,0.769131,0.775947,0.706734,0.626747,55.198537
2,vessel_run3_attentionunet_augON,attention_unet,True,18,16,0.208825,0.662644,0.973911,0.649905,0.735594,0.670161,0.594915,53.900351


In [30]:
!zip -r /kaggle/working/vessel_results.zip /kaggle/working/checkpoints /kaggle/working/logs /kaggle/working/runs_log_vessel.csv


  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/vessel_run1_unetplusplus_augOFF.pth (deflated 23%)
  adding: kaggle/working/checkpoints/vessel_run3_attentionunet_augON.pth (deflated 6%)
  adding: kaggle/working/checkpoints/vessel_run2_unetplusplus_augON.pth (deflated 22%)
  adding: kaggle/working/logs/ (stored 0%)
  adding: kaggle/working/logs/vessel_run3_attentionunet_augON.csv (deflated 50%)
  adding: kaggle/working/logs/vessel_run2_unetplusplus_augON.csv (deflated 50%)
  adding: kaggle/working/logs/vessel_run1_unetplusplus_augOFF.csv (deflated 50%)
  adding: kaggle/working/runs_log_vessel.csv (deflated 46%)
